# v043_stage1_m3 — M3's feature groups inside stage 1 of the two-stage matcher

| Field | Value |
|---|---|
| **Version** | `v043_stage1_m3` |
| **Plan group** | C5 (the stage-1 features behind the competition features), with M3's C2–C4 groups |
| **Parent version** | `v107_tight_rule` (v104's two-stage matcher, rule tuned on the tight mock) |
| **Sibling** | `v042_stage2_m3`: the same groups in stage 2 only; its arm A re-runs v107 on the same machine and is the decision baseline when present |
| **Author** | M3 |
| **Date** | 2026-09-26 |
| **Status** | kept |

v107 is a two-stage matcher. Stage 1 (v101: LightGBM on 53 pair features) scores every
candidate pair of the test-shaped mock fold and filters them; stage 2 (XGBoost on the GPU,
trained on the mock's fit entities) re-scores the kept pairs from their pair features plus
competition and anchor features built from stage 1's probability `p1`; the rule is tuned on
the tight mock. v043 changes stage 1 only: it is fitted exactly like v101, with M3's four
feature groups added. Everything downstream reads `p1`, so it is recomputed with v104's and
v107's settings rather than reused:

```
train fold ─► pipeline.fit as v101, on v101's 53 + M3's 23 features            ◄ this version
mock fold (as v104) ─► stage 1 on every pair: p1 ─► filter (p1 ≥ 0.01, top 16 per S1)
        ├─► competition features from p1 over all pairs, anchors over the kept pairs
kept pairs of the mock's fit entities ─► stage 2 (v104's configuration, cross-fitted)
stage-2 probs ─► pool-side 1-to-1 ─► rule tuned on the tight mock (v107) ─► est_public
```

The decision number is **est_public** on the mock val entities (TRACKER, "Tight mock": public
≈ 1 − L_FN − 1.45·L_FP − 0.0072, which reproduced uploads #2 and #3); plain mock F0.5 comes
second, and the plain-val F0.5 of the new stage 1 (§4.2) is a sanity check. House rules: every
code cell is preceded by a markdown cell saying what it does and why; every function has a
docstring.

## 1. Hypothesis

* **Change vs parent (v107):** stage 1 only. It is fitted like v101 (`pipeline.fit`: same
  blocking, the same 200k fit-side training entities, early stopping on 50k tune-side
  entities, LightGBM capped at 4,000 rounds, v101's rule grid on the tune side), with M3's
  groups after v101's 53 features: `idf` (8: idf-weighted name and address cosine, rarest
  shared token, coverage per side), `token_freq` (4: pool records sharing the exact name /
  address), `ctx_idf` (5: rank and gap of the idf cosines inside the S1 group, exact-name
  candidates) and `address_extra` (6: reverse containment, number containment, postcode
  prefix, empty S1 address, address length ratio): 76 stage-1 features. The mock fold, the
  filter, v104's stage-2 configuration (read from its `metrics.json`) and v107's rule tuning
  are unchanged. No `extra_groups`: every stage-1 feature is a column of the stage-2 frame,
  M3's included.
* **Why it should raise est_public:** in v104, stage 2 takes 72 % of its gain from `pool_gap`
  (p1 minus the best p1 another S1 entity gives the same record) and 19 % from `p1` itself;
  its pair features add little. Stage 2 is only as good as p1, and a better p1 improves every
  stage-2 input at once: the filter keeps the right records, `pool_gap` and `s1_gap` separate
  owners from rivals more cleanly, the anchors are the right records. As a single-stage model
  M3's groups gave **+0.0026 plain-val F0.5** (v040 0.9870 against v001 0.9844, every slice
  up, misses 53.9k → 46.2k, false merges 6.7k → 5.0k; `num_contain_l` #7 and `idf_addr_cos`
  #8 by gain). Appended to stage 2 only (v042) they refine a pair's own evidence but leave its
  rivals' p1, and so `pool_gap`, unchanged.
* **Expected effect:** plain-val F0.5 of the new stage 1 near or above v040's 0.9870 (v101:
  0.9858); on the mock, fewer misses at the tight rule's precision; est_public above the
  baseline by more than 0.001 (v107 logged: 0.9659).
* **Risks:** `token_freq` holds raw pool counts. Stage 1 learns them against the fit pool
  (~6.2M records, both countries) and reads them against the larger per-country mock and test
  pools (v040 §8 flagged the same shift); stage 2, trained at the mock's density, reads the
  same columns and can re-weight them. Cost: M3's groups on ~47M mock pairs slow the stage-1
  pass (v040 scored val 1.8× slower than v001 on the same machine).
* **Decision (§7):** baseline = arm A of v042 (v107's pipeline re-run on this machine) when
  its `metrics.json` exists, else v107's logged est_public. **KEEP** if est_public > baseline
  + 0.001, **DROP** if est_public ≤ baseline, **INVESTIGATE** in between.

## 2. Setup

All imports first (`display` is imported explicitly, so the code also runs outside Jupyter).
The next cell holds this version's parameters; the one after it derives the folders and
caches, rebuilds v104's stage-2 configuration from its logged record and reads the logged
references (v107, v104, v101, v040 and, if present, v042). Nothing heavy runs in §2.

In [1]:
import json
import shutil
import subprocess
import sys
import time
from dataclasses import asdict

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from entity_resolution import config as C
from entity_resolution.data import isin
from entity_resolution.decision import apply_rule, tune_expected
from entity_resolution.evaluate import blocking_report, error_samples
from entity_resolution.features import DEFAULT_GROUPS, FEATURE_COLUMNS, feature_names
from entity_resolution.mock import FP_WEIGHT, PUBLIC_OFFSET, build_mock, target_shape
from entity_resolution.model import MatcherParams
from entity_resolution.pipeline import (
    PipelineConfig,
    fit,
    mock_scores,
    peak_rss_gb,
    run_fold,
    tune_mock,
)
from entity_resolution.split import load_fold
from entity_resolution.stacking import ANCHOR_COLUMNS, STACK_COLUMNS
from entity_resolution.tracking import log_result, timed
from entity_resolution.trainset import inner_split, sample_s1
from entity_resolution.twostage import (
    TwoStage,
    TwoStageConfig,
    fit_stage2,
    mock_scored,
    mock_stage1,
    run_test_two_stage,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.max_columns", 30)
t_start = time.time()  # notebook wall time, printed at the end

### Parameters of this version (the only version-specific code cell)

`STAGE1_GROUPS` is v101's feature set (`DEFAULT_GROUPS` + `frequency`: 9 groups, 53 features)
followed by `M3_GROUPS` (23 features); `cfg` keeps every other `PipelineConfig` default, as
v101 did (blocking, the 200k fit-side sample, the rule grid), with v101's 4,000-round cap.
`BASELINE_VERSION` names the sibling whose arm A is the decision baseline when its
`metrics.json` exists. `RUN_VAL` scores the new stage 1 alone on the plain val fold (§4.2);
`RUN_TEST` stays `False` unless this version is shortlisted for an upload (§9).

In [2]:
VERSION = "v043_stage1_m3"         # this folder under experiments/
PARENT = "v107_tight_rule"         # logged version compared against (its metrics.json)
GROUP = "C5"                       # plan ID
OWNER = "M3"
M3_GROUPS = ("idf", "token_freq", "ctx_idf", "address_extra")
STAGE1_GROUPS = (*DEFAULT_GROUPS, "frequency", *M3_GROUPS)       # v101's groups + M3's
cfg = PipelineConfig(feature_groups=STAGE1_GROUPS, model=MatcherParams(n_estimators=4000))
BASELINE_VERSION = "v042_stage2_m3"  # same-machine baseline arm (metrics.json "arms"), if present
RUN_VAL = True    # plain-val score of the new stage 1 (single-stage, like v101's 0.9858)
RUN_TEST = False  # test inference only for shortlisted versions

Derived paths, the stage-2 configuration and the logged references:

* `tcfg` is v104's `TwoStageConfig`, rebuilt from its `metrics.json` record (`two_stage`:
  floor 0.01, top 16, 2 cross-fitting parts, seed 6161, 50k early-stopping entities, anchors
  on, cohesion off, XGBoost on the GPU with at most 4,000 rounds); list fields go back to
  tuples as in `TwoStage.load`. `extra_groups` stays empty: M3's columns are stage-1 features
  here and reach the stage-2 frame anyway (v042 tests them in stage 2 alone).
* `STAGE1_DIR` receives the fitted stage 1 (`Fitted.save`: model, rule, token map, config).
* `STAGE1_CACHE` holds the stage-1 outputs (kept pairs + stage-2 frame) of the mock and test
  partitions. Such a cache is valid for one stage-1 model, blocking configuration, filter and
  anchor switch, so its name carries this version's tag (its stage 1), the blocking key and
  the filter; v104's `v101_…` cache is never read. Delete it if stage 1 is refitted with
  changed code.
* References come from the logged `metrics.json` files, never retyped: v107's scores
  (`comparison["v107 tight rule"]`), v104's filter and stage-2 fit, v101's and v040's
  plain-val scores, and v042's arm A if present. The check below stops the notebook unless
  stage 1 is exactly v101's groups and LightGBM parameters plus `M3_GROUPS`, and stage 2
  v104's configuration without extra groups.

In [3]:
TAG, PARENT_TAG, BASE_TAG = (v.split("_")[0] for v in (VERSION, PARENT, BASELINE_VERSION))
EXP_DIR = C.EXPERIMENTS / VERSION
ARTIFACTS = EXP_DIR / "artifacts"          # gitignored: models, scored pairs
ARTIFACTS.mkdir(parents=True, exist_ok=True)
STAGE1_DIR = ARTIFACTS / "stage1"          # the fitted stage 1 (Fitted.save)


def logged(version: str) -> dict:
    """The ``metrics.json`` record of a logged version, or {} when the file is absent."""
    path = C.EXPERIMENTS / version / "metrics.json"
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


v104 = logged("v104_two_stage")["metrics"]
rec = v104["two_stage"]              # v104's TwoStageConfig; lists back to tuples (TwoStage.load)
tcfg = TwoStageConfig(**{**rec, "model": MatcherParams(**rec["model"]),
                         **{k: tuple(v) for k, v in rec.items() if isinstance(v, list)}})
STAGE1_CACHE = (cfg.cache_dir / "stage1" / f"{TAG}_{cfg.blocking.key()}_f{tcfg.floor}"
                f"_k{tcfg.max_cands}_a{int(tcfg.anchors)}")
parent = logged(PARENT)["metrics"]
parent_rec = parent["comparison"]["v107 tight rule"]    # f_beta, f_tight, est_public, ...
v101 = logged("v101_name_frequency")["metrics"]
v040 = logged("v040_idf_freq_features")["metrics"]
baseline_rec = logged(BASELINE_VERSION).get("metrics", {}).get("arms", {}).get("A: v104 columns")
M3_FEATURES = {c: g for g in M3_GROUPS for c in FEATURE_COLUMNS[g]}   # feature -> M3 group
timings: dict[str, float] = {}

# stage 1 = v101's logged groups and LightGBM parameters + M3_GROUPS; stage 2 = v104's
n_features = len(feature_names(cfg.feature_groups))   # raises on an unknown or repeated group
same_groups = [g for g in cfg.feature_groups if g not in M3_GROUPS] == v101["feature_groups"]
same_model = {k: v for k, v in asdict(cfg.model).items()
              if k in v101["model_params"]} == v101["model_params"]
if not (same_groups and same_model) or tcfg.extra_groups:
    raise ValueError("stage 1 must be v101's configuration + M3_GROUPS, stage 2 v104's")
print(f"{VERSION}: stage 1 = {n_features} features ({n_features - len(M3_FEATURES)} v101 + "
      f"{len(M3_FEATURES)} M3); stage-1 cache {STAGE1_CACHE.name}")
print("two-stage:", json.dumps(tcfg.record()))
print("FP_WEIGHT", FP_WEIGHT, "PUBLIC_OFFSET", PUBLIC_OFFSET)
print(f"parent {PARENT_TAG} (logged):", {k: round(v, 5) for k, v in parent_rec.items()})
print(f"baseline {BASE_TAG} arm A:", "absent: the parent's logged est_public decides"
      if baseline_rec is None else {k: baseline_rec.get(k) for k in ("est_public", "f_beta")})

v043_stage1_m3: stage 1 = 76 features (53 v101 + 23 M3); stage-1 cache v043_66540dae_f0.01_k16_a1
two-stage: {"floor": 0.01, "max_cands": 16, "folds": 2, "seed": 6161, "n_stop_s1": 50000, "anchors": true, "cohesion": false, "rivals": false, "train_roles": ["fit"], "model": {"backend": "xgb", "num_leaves": 63, "learning_rate": 0.05, "n_estimators": 4000, "early_stopping": 100, "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1, "min_data_in_leaf": 200, "lambda_l2": 1.0, "max_bin": 255, "scale_pos_weight": 1.0, "seed": 42, "num_threads": 12, "device": "cuda", "min_child_weight": 1.0}, "extra_groups": []}
FP_WEIGHT 1.45 PUBLIC_OFFSET 0.0072
parent v107 (logged): {'f_beta': 0.97451, 'f_tight': 0.97308, 'est_public': 0.96588, 'f_beta_singletons': 0.98367, 'pair_precision': 0.99648, 'pair_recall': 0.93423}
baseline v042 arm A: {'est_public': 0.9658784726663903, 'f_beta': 0.974510893391814}


## 3. Data

The mock fold exactly as in v104 and v107 (`build_mock`, seeds 5151 / 5152): per country the
test's pool size and pool-per-S1 ratio (India 710k S1 against a 4.13M pool, US 663k against
3.82M, ~40 % of the pool unowned as on test), the stage-1 training sample dropped first so its
records stay as unowned decoys, every val and tune entity of a kept cluster present. The
dropped sample, `sample_s1(fit side, 200k)`, is exactly the one `fit` trains on in §4.1
(`sample_s1` depends on ids only), so no present entity is one stage 1 trained on (tune
entities only drive its early stopping, as in v104). The mock's candidate pairs come from the
blocking cache when a two-stage version (v104, v042) already blocked this mock on the machine
(same blocking key, ids and token map); otherwise §4.3 blocks it once (~25 min on M1's
machine) and caches it. `train` stays loaded for the stage-1 fit (§4.1) and `val` for the
plain-val check (§4.2); the inner-split frames are released.

In [4]:
with timed("load", timings):
    train = load_fold("train", columns=[C.COUNTRY])
    val = load_fold("val", columns=[C.COUNTRY])
    fit_fold, tune_fold = inner_split(train)
    fit_sample = sample_s1(fit_fold.s1, cfg.n_fit_s1)[C.ENTITY_ID]
    mock = build_mock(train, val, tune_fold.s1[C.ENTITY_ID], fit_sample, target_shape())
del fit_fold, tune_fold, fit_sample      # train is kept for §4.1, val for §4.2
display(mock.info)
mock.fold.summary()

,country,s1_train,pool_train,keep_frac,s1_kept,pool_kept,s1_present,pool_per_s1,test_pool_per_s1,present_val,present_tune,present_fit
0,India,883188,4133346,1.000,883188,4133346,709678,5.824,5.824,176522,176208,356948
1,US,1323633,6186873,0.617,815850,3816702,663049,5.756,5.756,163562,163325,336162


{'fold': 'mock',
 's1': 1372727,
 's2': 3878856,
 's3': 4071192,
 'true_pairs': 4753992,
 'singleton_share': 0.0558}

## 4. Method

### 4.1 Stage 1: v101's fit with M3's groups

`pipeline.fit` on the train fold, as v101 ran it: the learned token map, the inner split
(seed 4242), 200k fit-side S1 entities blocked against the whole fit pool, early stopping on
50k tune-side entities, LightGBM (63 leaves, learning rate 0.05, at most 4,000 rounds), then
v101's rule grid on all tune-side entities (this plain rule is not used downstream; it makes
stage 1 a complete single-stage version for §4.2). `fit` reads only the fold's ids and truth
pairs: the country column loaded for the mock changes nothing, since samples and cache keys
depend on ids alone. M3's pool statistics (`idf`, `ctx_idf`, `token_freq`) are counted per
country over the pool each side is blocked against. The val fold is not touched.

Printed next to v101's logged fit (1,666 rounds; tune log-loss 0.00939 and AUC 0.99986 in its
notebook; tune-side F0.5 0.98623): the rounds, tune log-loss / AUC and the tune-side rule;
then M3's share of the gain, the stage-1 rank of every M3 feature and the gain top 25
(`m3_group` names the M3 group of a feature, empty for v101's features).

In [5]:
def ranked(importance: pd.Series) -> pd.DataFrame:
    """Gain share, rank (1 = largest share) and M3 group ("" for other features) per feature."""
    imp = importance.sort_values(ascending=False, kind="stable")
    return pd.DataFrame({"gain_share": imp.to_numpy(), "rank": np.arange(1, len(imp) + 1),
                         "m3_group": [M3_FEATURES.get(c, "") for c in imp.index]},
                        index=imp.index)


fit_timings: dict[str, float] = {}
t0 = time.time()
stage1 = fit(cfg, train, STAGE1_DIR, fit_timings)    # reads only ids and truth pairs of train
timings["stage1_fit_seconds"] = round(time.time() - t0, 2)
del train
info1 = stage1.info["fit_info"]
tune_f1 = float(stage1.tune_table["f_beta"].max())
print(f"fit() {timings['stage1_fit_seconds']:.0f} s; stages: {fit_timings}")
print(f"rounds {info1['best_iteration']} (v101 {v101['best_iteration']}); tune logloss "
      f"{info1['tune_logloss']:.5f}, AUC {info1['tune_auc']:.5f}; tune-side F0.5 "
      f"{tune_f1:.5f} (v101 {v101['tune_f_beta']:.5f})")
print("tune-side rule:", stage1.rule, "| v101:", v101["rule"])
rank1 = ranked(stage1.matcher.importance())
m3_share1 = rank1.groupby("m3_group")["gain_share"].sum().drop("", errors="ignore")
print(f"M3 share of stage-1 gain {m3_share1.sum():.4f}:",
      {g: round(float(s), 4) for g, s in m3_share1.items()})
print("stage-1 rank of every M3 feature:",
      {f: int(r) for f, r in rank1["rank"].reindex(list(M3_FEATURES)).items()})
rank1.head(25)

fit() 1478 s; stages: {'normalise_seconds': 6.65, 'fit_load_seconds': 48.64, 'fit_blocking_seconds': 8.94, 'stop_load_seconds': 18.87, 'stop_blocking_seconds': 2.71, 'features_seconds': 202.8, 'fit_seconds': 626.74, 'tune_load_seconds': 18.24, 'tune_blocking_seconds': 6.34, 'score_seconds': 491.16, 'tune_seconds': 26.74}
rounds 1671 (v101 1666); tune logloss 0.00787, AUC 0.99989; tune-side F0.5 0.98783 (v101 0.98623)
tune-side rule: DecisionRule(tau_abs=0.46, tau_rel=0.0, tau_single=0.46, max_matches=11, one_to_one=True) | v101: {'tau_abs': 0.42, 'tau_rel': 0.0, 'tau_single': 0.52, 'max_matches': 11, 'one_to_one': True}
M3 share of stage-1 gain 0.1423: {'address_extra': 0.0627, 'ctx_idf': 0.0154, 'idf': 0.0558, 'token_freq': 0.0085}
stage-1 rank of every M3 feature: {'idf_name_cos': 30, 'idf_name_top': 33, 'idf_name_cover_l': 28, 'idf_name_cover_r': 20, 'idf_addr_cos': 8, 'idf_addr_top': 39, 'idf_addr_cover_l': 43, 'idf_addr_cover_r': 17, 'freq_name_l': 55, 'freq_name_r': 36, 'freq_add

,gain_share,rank,m3_group
feature,,,
ad_token_set,0.338828,1,
sim_name_addr_word,0.147760,2,
ctx_gap_addr,0.067670,3,
core_token_set,0.060888,4,
core_jw,0.047504,5,
ctx_rank_addr,0.034805,6,
num_contain_l,0.034682,7,address_extra
idf_addr_cos,0.024178,8,idf
nm_token_sort,0.021748,9,


### 4.2 Plain-val check of the new stage 1 (`RUN_VAL`)

The new stage 1 alone, scored once on the fixed val fold with its own tune-side rule
(`run_fold`, v101's candidates from the blocking cache), next to v101 (0.9858) and v040
(v001 + M3's groups, 0.9870). A sanity check that M3's groups still add on top of v101's
`frequency` group, not the decision number: val is 3–6× less dense than test. `val` is
released afterwards.

In [6]:
VAL_KEYS = ["f_beta", "f_beta_singletons", "f_beta_matched", "pair_precision", "pair_recall",
            "cand_recall"]
val_metrics = None
if RUN_VAL:
    t0 = time.time()
    val_metrics = run_fold(cfg, stage1, val)[0]      # metrics only; the frames are dropped
    timings["val_seconds"] = round(time.time() - t0, 2)
    rows = {"v101 (logged)": v101, "v040 (logged: v001 + M3 groups)": v040,
            f"{TAG} stage 1": val_metrics}
    val_cmp = pd.DataFrame({name: {k: r.get(k) for k in VAL_KEYS}
                            for name, r in rows.items()}).T.astype(float)
    val_cmp["d_f_beta_vs_v101"] = val_cmp["f_beta"] - val_cmp.loc["v101 (logged)", "f_beta"]
    display(val_cmp.round(5))
del val

,f_beta,f_beta_singletons,f_beta_matched,pair_precision,pair_recall,cand_recall,d_f_beta_vs_v101
v101 (logged),0.98582,0.98740,0.98573,0.99657,0.96543,0.99057,0.00000
v040 (logged: v001 + M3 groups),0.98698,0.98752,0.98695,0.99641,0.96889,0.99057,0.00116
v043 stage 1,0.98756,0.98862,0.98750,0.99739,0.96852,0.99057,0.00174


### 4.3 Stage 1 on every mock pair: filter, competition and anchor features

`mock_stage1` builds the 76 stage-1 features chunk by chunk over each country's mock pairs
(~47M in all; blocked first if the blocking cache lacks them, see §3), scores them with the
new stage 1, keeps the pairs with
`p1 ≥ 0.01` among each entity's 16 best, and adds v104's competition features (over all pairs
of the country) and anchor features (over the kept pairs). M3's pool statistics are counted
once per country over the whole mock pool, as `run_test_two_stage` counts them over the test
pool. The outputs are cached under `STAGE1_CACHE / "mock"`. The report gives candidate recall
after the filter per role next to v104's (stage 1 = v101): a sharper p1 should keep more true
pairs at a similar candidate count.

In [7]:
t0 = time.time()
outs = mock_stage1(cfg, stage1, mock, tcfg, timings=timings, cache_dir=STAGE1_CACHE / "mock")
timings["stage1_seconds"] = round(time.time() - t0, 2)
kept = pd.concat([o.pairs for o in outs.values()], ignore_index=True)
n_cols = next(iter(outs.values())).X.shape[1]
print(f"stage 1 {timings['stage1_seconds']:.0f} s; pairs {sum(o.n_all for o in outs.values()):,}"
      f" -> kept {len(kept):,}; stage-2 frame {n_cols} columns")
filter_report = pd.DataFrame({role: blocking_report(kept, mock.part(role))
                              for role in ("fit", "tune", "val")}).T
del kept
print(f"v104 (stage 1 = v101), val entities: pair recall {v104['cand_recall_val']:.4f} at "
      f"{v104['cands_mean_val']:.2f} candidates per S1")
filter_report[["pair_recall", "entity_recall", "ceiling_f_beta", "candidates_mean",
               "candidates_p95", "candidates_max"]].round(4)

stage 1 2130 s; pairs 47,348,383 -> kept 6,011,983; stage-2 frame 93 columns


v104 (stage 1 = v101), val entities: pair recall 0.9644 at 4.59 candidates per S1


,pair_recall,entity_recall,ceiling_f_beta,candidates_mean,candidates_p95,candidates_max
fit,0.9641,0.9964,0.9874,4.3825,8.0,16.0
tune,0.9640,0.9963,0.9874,4.3784,8.0,16.0
val,0.9643,0.9964,0.9875,4.3748,8.0,16.0


### 4.4 Stage 2: v104's configuration on the new stage-1 frame

Two XGBoost models on the GPU, each trained on half of the mock's fit entities (by id hash)
with early stopping on the kept pairs of 50k tune entities; later, fit entities are scored by
the model that did not see them, tune and val entities by the mean of both. The frame holds
every stage-1 feature (M3's included), the twelve competition and the five anchor features.

Shown: each model's fit next to v104's (the early-stopping rows are the kept pairs of the same
tune entities, so the log-losses are nearly comparable; the filter changed which pairs are
kept); the gain by kind of feature (v104: about 95 % on the competition features); the gain
top 25; every M3 feature's rank and gain share in both stages.

In [8]:
t0 = time.time()
models, fit_info = fit_stage2(outs, mock, tcfg)
timings["fit_seconds"] = round(time.time() - t0, 2)
FIT_KEYS = ("rows", "positive_rate", "best_iteration", "tune_logloss", "tune_auc", "fit_seconds")
display(pd.DataFrame({f"{run} fold{f}": {k: info[f"fold{f}"].get(k) for k in FIT_KEYS}
                      for run, info in (("v104", v104["fit_info"]), (TAG, fit_info))
                      for f in range(tcfg.folds)}).T)
rank2 = ranked(pd.concat([m.importance() for m in models], axis=1).mean(axis=1))
feature_kind = np.select([rank2.index.isin(STACK_COLUMNS), rank2.index.isin(ANCHOR_COLUMNS),
                          (rank2["m3_group"] != "").to_numpy()],
                         ["competition", "anchor", "M3 pair feature"], "other pair feature")
gain_by_kind = rank2.groupby(feature_kind)["gain_share"].sum()
print("stage-2 gain by kind:", {k: round(float(v), 4) for k, v in gain_by_kind.items()})
display(rank2.head(25))
m3 = list(M3_FEATURES)
m3_ranks = pd.DataFrame({"group": [M3_FEATURES[c] for c in m3],
                         "rank_s1": rank1["rank"].reindex(m3).to_numpy(),
                         "gain_s1": rank1["gain_share"].reindex(m3).to_numpy(),
                         "rank_s2": rank2["rank"].reindex(m3).to_numpy(),
                         "gain_s2": rank2["gain_share"].reindex(m3).to_numpy()}, index=m3)
m3_ranks.sort_values("rank_s2")

,rows,positive_rate,best_iteration,tune_logloss,tune_auc,fit_seconds
v104 fold0,1595991.0,0.726037,1813.0,0.057458,0.996679,300.24
v104 fold1,1593066.0,0.726280,1645.0,0.057386,0.996703,275.76
v043 fold0,1520420.0,0.762028,1382.0,0.052071,0.996931,149.28
v043 fold1,1517138.0,0.762554,1715.0,0.051853,0.996973,189.66


stage-2 gain by kind: {'M3 pair feature': 0.0144, 'anchor': 0.0057, 'competition': 0.9513, 'other pair feature': 0.0286}


,gain_share,rank,m3_group
feature,,,
pool_gap,0.721140,1,
p1,0.165802,2,
s1_gap,0.041169,3,
s1_p1_sum,0.007709,4,
pool_p1_sum,0.004866,5,
pool_best_other,0.004291,6,
sim_name_addr_word,0.002036,7,
s1_best_other,0.001971,8,
pool_rank,0.001814,9,


,group,rank_s1,gain_s1,rank_s2,gain_s2
idf_name_top,idf,33,0.003307,13,1.492542e-03
idf_addr_top,idf,39,0.001728,19,1.155182e-03
idf_addr_cover_r,idf,17,0.009389,21,1.123927e-03
idf_addr_cos,idf,8,0.024178,22,1.056926e-03
idf_addr_cover_l,idf,43,0.001371,31,8.296382e-04
idf_name_cover_r,idf,20,0.007868,37,7.658526e-04
freq_addr_r,token_freq,25,0.004792,38,7.159610e-04
ctx_rank_idf_addr,ctx_idf,13,0.012712,39,7.137080e-04
ad_contain_r,address_extra,11,0.016647,44,6.520299e-04
ctx_n_same_name,ctx_idf,60,0.000396,45,6.380430e-04


### 4.5 Decision rule tuned on the tight mock (as v107)

Stage-2 probabilities for every kept pair (fit entities out of fold), the pool-side 1-to-1
across all present entities of each country, then the rows of the tune and val entities
(`mock_scored`), saved to `artifacts/mock_scored.parquet` for decision-layer work. v107's two
candidates, both tuned for the tight score (false merges ×1.45) on the mock's tune entities:
the threshold grid (`tune_mock`) and expected-F0.5 decoding (`tune_expected`, γ from 0.7 to
2.0, expected misses from 0 to 0.4); the better tight tune score is the rule. The tune scores
are printed next to v107's (same tune entities). The stage-1 outputs (`outs`, 2+ GB) are
released here; re-running §4.3 reloads them from the cache.

In [9]:
t0 = time.time()
scored = mock_scored(outs, models, mock, tcfg)[0]    # [1] would repeat the filter report
del outs                                             # 2+ GB; §4.3 reloads it from the cache
scored.to_parquet(ARTIFACTS / "mock_scored.parquet", index=False)   # for decision-layer work
rule_t, table_t = tune_mock(scored, mock, cfg.grid, fp_weight=FP_WEIGHT)
tune_part = mock.part("tune")
rows_t = scored[isin(scored[C.S1_ID], pd.Index(tune_part.s1[C.ENTITY_ID]))]
rule_e, table_e = tune_expected(rows_t, tune_part.s1[C.ENTITY_ID], tune_part.pairs,
                                gammas=(0.7, 0.85, 1.0, 1.2, 1.5, 2.0),
                                misses=(0.0, 0.05, 0.1, 0.2, 0.4), fp_weight=FP_WEIGHT)
best_t, best_e = table_t["f_beta"].max(), table_e["f_beta"].max()
rule = rule_e if best_e > best_t else rule_t
table = table_e if rule is rule_e else table_t
timings["tune_seconds"] = round(time.time() - t0, 2)
del rows_t, tune_part
print(f"{len(scored):,} scored tune + val pairs")
print(f"tight tune score: threshold {best_t:.5f} {rule_t}\n"
      f"                  expected  {best_e:.5f} {rule_e}")
print(f"v107 tight tune:  threshold {parent['tight_tune_threshold']:.5f}, expected "
      f"{parent['tight_tune_expected']:.5f}")
print("chosen:", rule, "| v107:", parent["rule"])

2,458,207 scored tune + val pairs
tight tune score: threshold 0.97521 DecisionRule(tau_abs=0.72, tau_rel=0.8, tau_single=0.72, max_matches=11, one_to_one=True)
                  expected  0.97521 ExpectedRule(gamma=1.5, miss=0.0, max_matches=11, one_to_one=True)
v107 tight tune:  threshold 0.97310, expected 0.97311
chosen: DecisionRule(tau_abs=0.72, tau_rel=0.8, tau_single=0.72, max_matches=11, one_to_one=True) | v107: {'gamma': 1.5, 'miss': 0.05, 'max_matches': 11, 'one_to_one': True}


## 5. Evaluation on the mock val entities

### 5.1 Scores, all and per country

`mock_scores` of the chosen rule on the mock's val entities: plain mock F0.5, the tight score
(false merges ×1.45) and **est_public** (tight − 0.0072), singletons, pair precision and
recall, with the change in est_public and mock F0.5 against v107's logged scores per country.

In [10]:
res = mock_scores(scored, mock, rule)
cols = ["f_beta", "f_tight", "est_public", "f_beta_singletons", "pair_precision", "pair_recall"]
parent_by = pd.DataFrame(parent.get("by_country", {})).T.reindex(index=res.index, columns=cols)
shown = res[cols].astype(float)
for c in ("est_public", "f_beta"):
    shown[f"d_{c}_vs_{PARENT_TAG}"] = shown[c] - parent_by[c].astype(float)
shown.round(5)

,f_beta,f_tight,est_public,f_beta_singletons,pair_precision,pair_recall,d_est_public_vs_v107,d_f_beta_vs_v107
all,0.97618,0.97502,0.96782,0.99021,0.99694,0.93956,0.00194,0.00167
India,0.97206,0.97087,0.96367,0.98962,0.99687,0.92897,0.00225,0.00191
US,0.98062,0.97950,0.97230,0.99086,0.99701,0.95100,0.00160,0.00140


### 5.2 Against the parent and the same-machine baseline

v107 as logged (stage 1 = v101, run on M1's machine), v042's arm A when its `metrics.json`
exists (v104's columns and v107's rule tuning re-run by v042: stage 2 trains on the GPU, and a
same-machine re-run separates this change from machine noise; it is a fair baseline only if
v042 ran on this machine), this version, and this version's deltas against each.
`baseline_est`, the est_public §7 decides on, is arm A's when available, else v107's.

In [11]:
refs = {f"{PARENT_TAG} (logged)": parent_rec}
if baseline_rec is not None:
    refs[f"{BASE_TAG} arm A (same machine)"] = baseline_rec
cmp = pd.DataFrame({**{name: {c: r.get(c) for c in cols} for name, r in refs.items()},
                    TAG: res.loc["all", cols].to_dict()}).T.astype(float)
for name in refs:
    cmp.loc[f"{TAG} - {name.split()[0]}"] = cmp.loc[TAG] - cmp.loc[name]
arm_est = None if baseline_rec is None else baseline_rec.get("est_public")
if arm_est is not None:          # the same-machine arm decides when v042 logged it
    baseline_name, baseline_est = f"{BASE_TAG} arm A (same machine)", float(arm_est)
else:
    baseline_name, baseline_est = f"{PARENT_TAG} (logged)", float(parent_rec["est_public"])
print(f"decision baseline: {baseline_name}, est_public {baseline_est:.5f}")
cmp.round(5)

decision baseline: v042 arm A (same machine), est_public 0.96588


,f_beta,f_tight,est_public,f_beta_singletons,pair_precision,pair_recall
v107 (logged),0.97451,0.97308,0.96588,0.98367,0.99648,0.93423
v042 arm A (same machine),0.97451,0.97308,0.96588,0.98367,0.99648,0.93423
v043,0.97618,0.97502,0.96782,0.99021,0.99694,0.93956
v043 - v107,0.00167,0.00194,0.00194,0.00654,0.00046,0.00532
v043 - v042,0.00167,0.00194,0.00194,0.00654,0.00046,0.00532


## 6. Error analysis

Error counts on the mock val entities under the chosen rule (`apply_rule` on their rows after
the 1-to-1), next to v107's logged counts: false merges (a predicted pair that is not true, of
an entity with true matches), misses (a true pair not predicted while its entity predicted
something), false singletons (the true pairs of a matched entity predicted empty) and
singleton merges (predictions on a true singleton). Then 10 samples of the false merges, the
singleton merges and the misses with both raw records side by side: `with_raw` (v104's
helper) reads the raw name and address of the sampled ids from the Parquet cache, since the
mock fold was loaded without them. The pattern of the remaining errors picks the next version.

In [12]:
def with_raw(sample: pd.DataFrame) -> pd.DataFrame:
    """Raw name / address of both sides from the Parquet cache (only the sampled ids)."""
    ids = pd.concat([sample[C.S1_ID], sample[C.ENTITY_ID]]).unique().tolist()
    raw = pd.concat([pq.read_table(C.DATASET / ".cache" / f"train_source{s}.parquet",
                                   columns=[C.ENTITY_ID, C.NAME, C.ADDRESS],
                                   filters=[(C.ENTITY_ID, "in", ids)]).to_pandas()
                     for s in C.SOURCES]).set_index(C.ENTITY_ID)
    return sample.assign(name_l=sample[C.S1_ID].map(raw[C.NAME]),
                         addr_l=sample[C.S1_ID].map(raw[C.ADDRESS]),
                         name_r=sample[C.ENTITY_ID].map(raw[C.NAME]),
                         addr_r=sample[C.ENTITY_ID].map(raw[C.ADDRESS]))


KINDS = ("false_merge", "missed", "false_singleton", "singleton_merge")
part = mock.part("val")
matches = apply_rule(scored[isin(scored[C.S1_ID], pd.Index(part.s1[C.ENTITY_ID]))], rule)
counts = {k: len(error_samples(matches, part, k, n=10**9)) for k in KINDS}
parent_counts = parent.get("errors_mock", {}).get("v107 tight rule", {})
display(pd.DataFrame({f"{PARENT_TAG} (logged)": {k: parent_counts.get(k) for k in KINDS},
                      TAG: counts}))
for kind in ("false_merge", "singleton_merge", "missed"):
    print(f"--- {kind}: 10 of {counts[kind]:,} pairs")
    display(with_raw(error_samples(matches, part, kind, n=10, scored=scored)))

,v107 (logged),v043
false_merge,3522,3182
missed,74223,67844
false_singleton,3154,3270
singleton_merge,362,214


--- false_merge: 10 of 3,182 pairs


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-107585328,S2-474103230,0.906573,Simba International Ltd,"New Delhi, B-4/13, 2Nd Floor, Main Wali Nagar,...",Simba International,
1,S1-137695892,S3-697508841,0.860655,Classic Suisse LLC,"Phoenix, 15801 48th Street, AZ, Unit 1216",Classic Suisse LLC,
2,S1-176450615,S3-332436853,0.845847,Gurgaon India Private Limited,"Gurgaon, Dlf Building No.9, Haryana, Tower-A, ...",Gurgaon India Private Ltd,"B-03, Gurgaon, Gurugram, HR"
3,S1-361975205,S2-766494919,0.904299,Bangalore South Systems Private Limited,"#004, Ashirwadh, 13Th Cross, Bangalore South, ...",Private Bangal0re South Systems Limited,"Karnataka, BANGALORE, BANGALORE SOUTH, NO. 110/1"
4,S1-734849917,S3-835178366,0.887449,"Frontier Utility, LLC","Chesapeake City, Unit F, 301 Wimbledon Chase, VA","Frontier Útility, LLC",
5,S1-738128729,S2-959360977,0.988744,Faclara Capital Partners LLC,"8 Web Road, Georgetown, MA",Faclara Capital Partners,"29 WEB ROAD, MA, GEORGETOWN"
6,S1-835179861,S2-90631569,0.887332,Premier Global LLP,"505, Gf Nyay Khand-3, Indirapuram, Ghaziabad, ...",PREMIER SERVICES LLP,"509, GF NYAY KHAND-3, INDIRAPURAM, Uttar Pradesh"
7,S1-893959950,S3-746174645,0.923746,Vikram & Partners,"3/17 Azadgarh, Kolkata, Howrah, West Bengal",Vikram Vikram Partners,
8,S1-89401279,S3-692503117,0.881879,Technology Mnr India Pvt Ltd,"Transcon Triumph 704, Tower A Off New Link Roa...",Technology Mnr Infra Pvt Ltd,"Transcon Triumph 2-711, Mumbai, Mumbai City, MH"
9,S1-901935102,S3-980883204,0.844027,NQ Producer Pvt. Ltd.,"Khno-667 & 496, East Jawahar Nagar, Ghaziabad,...",Nq Producer Pvt Ltd,


--- singleton_merge: 10 of 214 pairs


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-113976931,S3-868477138,0.775903,East Delhi Life Private Limited,"G 306 Ground Floor Old No, Ghazipur, East Delh...",Private EAST Delhi LIFE Limited,"5921-A, East Delhi, Delhi, DL"
1,S1-143794032,S3-497831350,0.975585,Electrum Clinic,"Ramesh Nilaya, 1St Cross, K R Extension, Tumku...",Electrum Projects,"Door No 45 Ramesh Nilaya, 1St Cross, K R Exten..."
2,S1-181116332,S2-765632443,0.773857,Smyrna Animal Hospital Inc.,"110 Creek Court, Smyrna, TN",Smyrna Animal Hospital Inc,"TN, SMYRNA, 114 CREEK CT"
3,S1-30056058,S2-957804806,0.996502,Osprey Group,"231 Silvermine Avenue, Norwalk, CT",Osprey Group,"232 Silvermine Ave, NORWALK, CT"
4,S1-369692708,S2-534818635,0.973552,Aqube (India) Human South 24 Parganas,"South 24 Parganas, Tentul Baria, West Bengal, ...",Aqube (India) Technologies South 24,"পশ্চিমবঙ্গ, H.NO 27 TENTUL BARIA, 3RD FLOOR, M..."
5,S1-521952472,S3-623455304,0.784254,Msme Partnership Limited,C/O Anupam Vashney Village Ruheri Opposi Te Ma...,Limited Msme Trading,No 38- C/o Anupam Vashney Village Ruheri Oppos...
6,S1-5735,S2-127034641,0.775724,MG Exim Private Limited,5 Zainab Baug Buildingbharucha Road Dahisar Ea...,CMG Exim Private Limited #31084,NO 5 ZAINAB BAUG BUILDINGBHARUCHA ROAD DAHISAR...
7,S1-576813879,S2-871234060,0.869353,C 2 L Enbridge Inc,"309 Sunflower Dr, Fairfax, IA",C 2 L INC PARTNERS,
8,S1-581904824,S2-638404212,0.875816,AC Processing Pvt Ltd,"Delhi, Chamber-8, B-4/3, Model Townist",AC PRHGOSSING PVT (LTD),
9,S1-602701852,S2-661502079,0.870597,Brightan Dogecoin LLC,"15 Eric Clauson Lane, Falmouth, MA",Brrightan Dogecoin Llc - 4697928860,"0015 ERIC CLAUSON LN, FALMOUHT CDP, MA"


--- missed: 10 of 67,844 pairs


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-107161626,S3-749679468,NaN,A Cure 2 You,"105 Crum Road, Yuma, TN",Cure2you.Com,"10 Crum Rd, PMB 1253, Yuma, Tennessee"
1,S1-15307012,S3-646681622,NaN,Pioneer Care Private Limited,"Bangalore, No.78, Karnataka, K.No.293, Kaverap...",Pioneer Care Private Ltd,"H.no 78, Bangalore, KA"
2,S1-274554043,S2-843741800,NaN,Baba Food,"Office No-219, 11/5 Second Floor South Tukogan...",बाबा फूड,"277 OFICE NO-219, INDORE, NULL, Madhya Pradesh"
3,S1-583575761,S3-696689157,NaN,Ranjit & Brothers Corporation,"Madhav Heritage, 1641 Sadashiv Peth, Tilak Roa...",Ranjit &,"Madhav Heritage, Pune, NULL, MH"
4,S1-634305339,S2-131572980,NaN,Universal Exports Private Limited,"C-1, G-11, Ground Floor, Krishna Apra Plaza, S...",यूनिवर्सल एक्सपोर्ट्स प्राइवेट लिमिटेड,"C-##1, LUCKNOW HQ REGION, उत्तर प्रदेश"
5,S1-718326235,S3-740299382,NaN,HHD Beverages Private Limited,"1/25, Hazuri Bhawan, Peepal Mandi Road, Agra, ...",Ariapyra,"Agra, 1/25, UP, Agra"
6,S1-825888031,S3-345500730,NaN,Kolkata Service,"Howrah, West Bengal, Kolkata, Kaliprasanna Roy...",kolkataservice.com,"33, Howrah, Kolkata, WB"
7,S1-835179861,S2-817775873,NaN,Premier Global LLP,"505, Gf Nyay Khand-3, Indirapuram, Ghaziabad, ...",प्रीमियर ग्लोबल एलएलपी,"GF NYAY KHAND-3, Uttar Pradesh, PLOT 665 505, ..."
8,S1-857894194,S2-660202965,NaN,"Sephira R. Nunez, Esq.","Groton, MA, Unit F, 12 Brookfield Drive",sephirarnunez.com,"MA, 12A BROOKFIELD DR, GROTON"
9,S1-98315586,S2-315720146,0.713596,Delta Homecare,"1225 Osteen Street, Unit 1, Vidor, TX",Umbraectoorbi,"1225 OSTEEN ST, VIDOR, TX"


## 7. Log the result

Saves the two-stage version (`TwoStage.save`: stage 1, both stage-2 models, filter settings,
rule, tuning table) to `artifacts/two_stage`, then records `metrics.json` and this version's
row in `experiments/experiments.csv`, stamped with the git commit of `src/` (it must not end in
`-dirty`: commit library changes before running). `mock_f05` is the plain mock F0.5 of the val
entities; `local_f05` stays empty (a two-stage version is not a plain-val model: the stage-1
val score is in the record and the notes), and so does `cand_recall`. The decision is
computed, never typed, against `baseline_est` (§5.2):

* **KEEP**: est_public > baseline + 0.001;
* **DROP**: est_public ≤ baseline;
* **INVESTIGATE**: in between.

In [13]:
ts = TwoStage(stage1, models, rule, tcfg, table,
              {"fit": fit_info, "filter_report": filter_report.to_dict("index"),
               "fp_weight": FP_WEIGHT, "stage1_groups": list(cfg.feature_groups)})
ts.save(ARTIFACTS / "two_stage")
est = float(res.loc["all", "est_public"])
DECISION = ("KEEP" if est > baseline_est + 0.001
            else "DROP" if est <= baseline_est else "INVESTIGATE")
record = {
    "hypothesis": "M3's groups inside stage 1 sharpen p1 and with it the competition features "
                  "stage 2 lives on, raising est_public by > 0.001 over the same two-stage "
                  "pipeline with v101 as stage 1",
    "stage1_feature_groups": list(cfg.feature_groups), "m3_groups": list(M3_GROUPS),
    "stage1_model_params": asdict(cfg.model), "two_stage": tcfg.record(),
    "fp_weight": FP_WEIGHT, "public_offset": PUBLIC_OFFSET,
    "rule": asdict(rule), "rule_kind": type(rule).__name__,
    "tight_tune_threshold": float(best_t), "tight_tune_expected": float(best_e),
    "mock_f_beta": float(res.loc["all", "f_beta"]),
    "f_tight": float(res.loc["all", "f_tight"]), "est_public": est,
    **{f"mock_{c}": float(res.loc[c, "f_beta"]) for c in res.index if c != "all"},
    **{k: float(res.loc["all", k]) for k in ("f_beta_singletons", "f_beta_matched",
                                             "pair_precision", "pair_recall")},
    "by_country": res[cols].to_dict("index"),
    "val_f_beta": None if val_metrics is None else float(val_metrics["f_beta"]),
    "val_metrics": None if val_metrics is None else {k: val_metrics[k] for k in VAL_KEYS},
    "stage1_fit": {"fit_info": info1, "rule": asdict(stage1.rule), "tune_f_beta": tune_f1,
                   "token_map_size": stage1.info["token_map_size"], "timings": fit_timings},
    "stage1_importance_top25": rank1["gain_share"].head(25).to_dict(),
    "m3_share_stage1": m3_share1.to_dict(),
    "stage2_importance_top25": rank2["gain_share"].head(25).to_dict(),
    "stage2_gain_by_kind": gain_by_kind.to_dict(), "m3_ranks": m3_ranks.to_dict("index"),
    "cand_recall_val": float(filter_report.loc["val", "pair_recall"]),
    "cands_mean_val": float(filter_report.loc["val", "candidates_mean"]),
    "filter_report": filter_report.to_dict("index"), "fit_info": fit_info,
    "errors_mock": counts, "comparison": cmp.to_dict("index"),
    "baseline": {"name": baseline_name, "est_public": baseline_est},
    **timings, "peak_rss_gb": peak_rss_gb(), "decision": DECISION,
}
print(f"est_public {est:.5f} vs {baseline_name} {baseline_est:.5f} ({est - baseline_est:+.5f};"
      f" KEEP needs > +0.001) -> {DECISION}")
val_note = "" if val_metrics is None else f"; stage-1 val {val_metrics['f_beta']:.4f}"
row = log_result(
    EXP_DIR, change="stage 1 = v101 + M3 groups (idf, token_freq, ctx_idf, address_extra); "
                    "v104 two-stage; v107 tight rule",
    group=GROUP, mock_f05=float(res.loc["all", "f_beta"]), cand_recall=None,
    notes=(f"est_public {est:.4f}; baseline {baseline_name} {baseline_est:.4f}; "
           f"f_tight {res.loc['all', 'f_tight']:.4f}{val_note}"),
    metrics=record, owner=OWNER, parent=PARENT_TAG, decision=DECISION)
print(row)

est_public 0.96782 vs v042 arm A (same machine) 0.96588 (+0.00194; KEEP needs > +0.001) -> KEEP
{'version': 'v043', 'date': '2026-09-26', 'group': 'C5', 'change': 'stage 1 = v101 + M3 groups (idf, token_freq, ctx_idf, address_extra); v104 two-stage; v107 tight rule', 'local_f05': '', 'mock_f05': '0.9762', 'cand_recall': '', 'public_f05': '', 'commit': '024b47c', 'notes': 'est_public 0.9678; baseline v042 arm A (same machine) 0.9659; f_tight 0.9750; stage-1 val 0.9876', 'owner': 'M3', 'parent': 'v107', 'decision': 'KEEP'}


## 8. Conclusion

* **Result (§5):** est_public **0.96588 → 0.96782 (+0.00194)** and mock F0.5 0.97451 → 0.97618
  (+0.00167) against the same-machine baseline (v042 arm A, which reproduces the logged v107
  exactly). Singletons 0.98367 → 0.99021, pair precision 0.99648 → 0.99694, pair recall
  0.93423 → 0.93956; India +0.0023, US +0.0016. Stage 1 alone scores plain val **0.98756**
  (v101 0.98582, v040 0.98698): the best single-stage model so far.
* **Decision (§7): KEEP** (+0.00194 > 0.001).
* **Against v042** (M3's groups in stage 2 only): est_public 0.96782 against 0.96788, a tie.
  v043 finds more matches (misses 67,844 against 69,866; singletons 0.9902 against 0.9866) and
  merges more (false merges 3,182 against 2,744).
* **M3 features:** 14.2 % of stage 1's gain (address_extra 6.3 %, idf 5.6 %, ctx_idf 1.5 %,
  token_freq 0.9 %); `num_contain_l` #7, `idf_addr_cos` #8, `ad_contain_r` #11,
  `ctx_rank_idf_addr` #13. Tune logloss 0.00939 → 0.00787 (−16 %). The sharper p1 lets the
  filter keep 4.37 candidates per S1 instead of 4.59 at the same recall (0.9643). In stage 2
  the competition features take 95 % of the gain (`pool_gap` 72 %), M3's columns 1.4 %.
* **Errors (§6):** false merges 3,522 → 3,182, misses 74,223 → 67,844, false singletons
  3,154 → 3,270, singleton merges 362 → 214. Remaining false merges pair a full record with a
  name-only pool record of a close name (`Simba International Ltd` / `Simba International`,
  empty address); misses are mostly scripts and domain forms outside the kept candidates.
* **Cost:** stage-1 fit 1,478 s, mock stage-1 pass 2,130 s (76 features on 47.3M pairs),
  stage 2 ~5.6 min on the RTX 2050; peak RSS 5.1 GB.
* **Next:** v042 and v043 tie on est_public; v042 goes to test inference (fewer false merges,
  v101's public-proven stage 1 unchanged, cheaper at test scale). Upload is M1's decision.


## 9. Test inference (shortlisted versions only)

Runs only with `RUN_TEST = True` (parameter cell), set when this version is shortlisted for
an upload; otherwise the cell writes nothing. `run_test_two_stage`, one country at a time
(France included): blocking (cached), the new stage 1 with the filter and the competition and
anchor features (cached under `STAGE1_CACHE / "test"`), stage 2 (mean of both models), the
rule; `candidate_pairs.tsv` holds exactly the filtered pairs stage 2 scored. Then the
per-country sanity table, a copy of both files in `submissions/v043/`, and both validators
(ours with id checks, then the organisers').

In [14]:
if RUN_TEST:
    t0 = time.time()
    match_path, cand_path, s1n_test, test_matches, test_summary = run_test_two_stage(
        cfg, ts, cache_dir=STAGE1_CACHE / "test")
    print(f"run_test {time.time() - t0:.0f} s -> {match_path}, {cand_path}")
    country_of = s1n_test.set_index(C.ENTITY_ID)[C.COUNTRY]
    n_s1 = s1n_test.groupby(C.COUNTRY).size()
    by = test_matches[C.S1_ID].map(country_of)
    display(pd.DataFrame({
        "s1": n_s1,
        "cands_per_s1": test_summary["n_cands"].groupby(
            test_summary.index.map(country_of)).sum() / n_s1,
        "matched_share": test_matches.groupby(by)[C.S1_ID].nunique() / n_s1,
        "matches_per_s1": test_matches.groupby(by).size() / n_s1}))
    dest = C.ROOT / "submissions" / TAG
    dest.mkdir(parents=True, exist_ok=True)
    for p in (match_path, cand_path):
        shutil.copy2(p, dest / p.name)
    print("copied to", dest)
    out = subprocess.run([sys.executable, "-m", "entity_resolution.submission", "--output-dir",
                          str(C.OUTPUT), "--check-ids"], capture_output=True, text=True)
    print(out.stdout[-2000:], out.stderr[-2000:])
    out = subprocess.run([sys.executable, str(C.OFFICIAL_VALIDATOR), "--matching",
                          str(match_path), "--candidate", str(cand_path), "--test-dir",
                          str(C.DATASET / "test")], capture_output=True, text=True)
    print(out.stdout[-3000:], out.stderr[-2000:])
else:
    print("RUN_TEST is False: no test inference (set it in the parameter cell to shortlist)")
print(f"notebook total {time.time() - t_start:.0f} s, peak RSS {peak_rss_gb()} GB")

RUN_TEST is False: no test inference (set it in the parameter cell to shortlist)
notebook total 4744 s, peak RSS 5.11 GB
